# Tutorial 2: Spatial Visualization of IMC Tissue Data

**Systems Biology — Spatial Proteomics Module**

## Learning Objectives
1. Understand how spatial coordinates are stored and prepared for spatial tools
2. Visualise tissue architecture by plotting cells at their physical positions
3. Overlay protein marker expression on spatial maps
4. Quantify immune infiltration by computing distances between cell types
5. Compare spatial organisation across images and cancer types

## Why does spatial context matter?

Single-cell expression tells us **what** each cell is; spatial coordinates tell us **where** it is in the tissue.  
The distance between a CD8+ T cell and a tumour cell is a direct physical readout of the immune response:  
- **Immune-infiltrated (hot) tumours**: CD8 cells intermingle with tumour cells → better prognosis  
- **Immune-excluded tumours**: T cells are trapped at the tumour border  
- **Immune-desert tumours**: Very few T cells anywhere in the image  

---

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial import KDTree
import seaborn as sns

plt.rcParams['figure.dpi'] = 100
sns.set_style('white')

DATA_PATH = 'data/train_adata.h5ad'

---
## Part 1: Loading Data and Setting Up Spatial Coordinates

IMC spatial coordinates are stored in `.obs['Pos_X']` and `.obs['Pos_Y']` (micrometers).  
Spatial analysis tools (Squidpy, etc.) expect them in `.obsm['spatial']` as a NumPy array of shape `(n_cells, 2)`.

In [ ]:
adata = ad.read_h5ad(DATA_PATH)

# Create arcsinh layer for marker expression visualisation
if 'exprs_arcsinh' not in adata.layers:
    adata.layers['exprs_arcsinh'] = np.arcsinh(adata.layers['exprs'] / 5)

# Store spatial coordinates in .obsm['spatial'] (required by Squidpy)
adata.obsm['spatial'] = adata.obs[['Pos_X', 'Pos_Y']].values

print(f"Cells: {adata.n_obs:,}  |  Images: {adata.obs['image'].nunique()}")
print(f"Spatial coords shape: {adata.obsm['spatial'].shape}")
print(f"X range: [{adata.obs['Pos_X'].min():.1f}, {adata.obs['Pos_X'].max():.1f}] um")
print(f"Y range: [{adata.obs['Pos_Y'].min():.1f}, {adata.obs['Pos_Y'].max():.1f}] um")

In [ ]:
# Physical image size overview
# IMC images are typically 600x600 um (1 pixel ~ 1 um)
img_stats = adata.obs.groupby('image', observed=True).agg(
    n_cells=('Pos_X', 'count'),
    x_range=('Pos_X', lambda x: x.max() - x.min()),
    y_range=('Pos_Y', lambda x: x.max() - x.min()),
    indication=('Indication', 'first')
)
print("Image size and cell count statistics:")
print(img_stats[['n_cells', 'x_range', 'y_range']].describe().round(1).to_string())
print("\nImages with most cells:")
print(img_stats.nlargest(5, 'n_cells')[['n_cells', 'indication']].to_string())

---
## Part 2: Visualising a Single Tissue Image

We build up the visualisation layer by layer:
1. All cells as grey dots (tissue shape)
2. Colour by cell type (tissue architecture)

In [ ]:
# Select the largest image (most cells)
img_sizes = adata.obs.groupby('image', observed=True).size()
IMAGE_ID = img_sizes.idxmax()
print(f"Selected image: {IMAGE_ID}")
print(f"({img_sizes.max()} cells)")

adata_img = adata[adata.obs['image'] == IMAGE_ID].copy()
coords = adata_img.obsm['spatial']
print(f"\nCell types present:")
print(adata_img.obs['celltypes'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# All cells, grey
axes[0].scatter(coords[:, 0], coords[:, 1], s=2, alpha=0.4, color='dimgrey')
axes[0].set_aspect('equal')
axes[0].set_title('All cells - spatial layout')
axes[0].set_xlabel('X (um)')
axes[0].set_ylabel('Y (um)')
axes[0].invert_yaxis()

# Colour by cell type
cell_types = sorted(adata_img.obs['celltypes'].unique())
palette = dict(zip(cell_types, sns.color_palette('tab20', len(cell_types))))

for ct in cell_types:
    mask = adata_img.obs['celltypes'] == ct
    c = coords[mask]
    axes[1].scatter(c[:, 0], c[:, 1], s=3, alpha=0.6,
                    color=palette[ct], label=ct, rasterized=True)

axes[1].set_aspect('equal')
axes[1].set_title('Cells coloured by type')
axes[1].set_xlabel('X (um)')
axes[1].set_ylabel('Y (um)')
axes[1].invert_yaxis()
axes[1].legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8,
               markerscale=4, title='Cell type')

plt.suptitle(f'Tissue image overview\n{IMAGE_ID[-60:]}', fontsize=10)
plt.tight_layout()
plt.show()

---
## Part 3: The Tumour Microenvironment

The most clinically relevant spatial question: **where are immune cells relative to tumour cells?**

We overlay:
- Tumour cells (red)
- CD8+ cytotoxic T cells (blue) — the primary anti-tumour killers
- MacCD163 macrophages (green) — often tumour-promoting
- All other cells (light grey) for tissue context

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

highlight_types = ['Tumor', 'CD8', 'MacCD163']

for ax_idx, (ax, focus_ct, focus_col, focus_label) in enumerate(zip(
    axes,
    ['CD8',        'MacCD163'],
    ['royalblue',  'forestgreen'],
    ['CD8+ T cell', 'Macrophage CD163']
)):
    # Background: non-highlighted cells
    bg_mask = ~adata_img.obs['celltypes'].isin(highlight_types)
    ax.scatter(coords[bg_mask, 0], coords[bg_mask, 1],
               s=1, alpha=0.15, color='lightgrey', rasterized=True)

    # Tumour cells
    tm = adata_img.obs['celltypes'] == 'Tumor'
    ax.scatter(coords[tm, 0], coords[tm, 1],
               s=4, alpha=0.7, color='crimson', label='Tumor', rasterized=True)

    # Focus cell type
    fm = adata_img.obs['celltypes'] == focus_ct
    ax.scatter(coords[fm, 0], coords[fm, 1],
               s=7, alpha=0.85, color=focus_col, label=focus_label,
               zorder=3, rasterized=True)

    ax.set_aspect('equal')
    ax.set_title(f'Tumor vs {focus_label}', fontsize=12)
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')
    ax.invert_yaxis()
    ax.legend(loc='lower right', fontsize=9, markerscale=3)

plt.suptitle('Tumour microenvironment spatial layout', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 2.1: Spatial Pattern Observation

**Question:** Looking at the plots above, describe the spatial relationship between:
1. Tumour cells and CD8+ T cells - are they intermixed or segregated?
2. Tumour cells and macrophages - do macrophages surround the tumour or avoid it?

**Task:** Choose a different image (e.g., the second or third largest) and make the same overlay. Is the pattern the same?

In [ ]:
# Your code here - pick a different image
IMAGE_ID2 = img_sizes.nlargest(10).index[3]
print(f"Exploring: {IMAGE_ID2}")
print(f"Indication: {adata.obs.loc[adata.obs['image']==IMAGE_ID2, 'Indication'].iloc[0]}")

adata_img2 = adata[adata.obs['image'] == IMAGE_ID2].copy()
c2 = adata_img2.obsm['spatial']

fig, ax = plt.subplots(figsize=(8, 8))
bg2 = ~adata_img2.obs['celltypes'].isin(['Tumor', 'CD8'])
ax.scatter(c2[bg2, 0], c2[bg2, 1], s=1, alpha=0.1, color='lightgrey')
for ct, col in [('Tumor', 'crimson'), ('CD8', 'royalblue')]:
    m = adata_img2.obs['celltypes'] == ct
    ax.scatter(c2[m, 0], c2[m, 1], s=5, alpha=0.8, color=col, label=ct)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend(markerscale=3, fontsize=10)
ax.set_title(f'Tumor vs CD8 - second image\n({IMAGE_ID2[-50:]})')
plt.tight_layout()
plt.show()

---
## Part 4: Marker Expression in Spatial Context

Beyond cell type labels, we can map **continuous marker expression** onto spatial coordinates.

Markers of interest:
- **E-cadherin (Ecad)**: Epithelial adhesion protein, marks tumour cells
- **PD1**: Immune checkpoint on exhausted T cells - key immunotherapy target
- **PD-L1 (PDL1)**: PD1 ligand on tumour/macrophages - inhibits T cell killing
- **Granzyme B (GrzB)**: Cytotoxic molecule in active CD8/NK cells

In [ ]:
markers_to_map = ['Ecad', 'PD1', 'PDL1', 'GrzB']
marker_labels = {
    'Ecad': 'E-cadherin (tumour marker)',
    'PD1':  'PD1 (T cell exhaustion)',
    'PDL1': 'PD-L1 (immune suppression)',
    'GrzB': 'Granzyme B (cytotoxic activity)',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, mname in zip(axes, markers_to_map):
    midx = np.where(adata_img.var['marker'] == mname)[0]
    if len(midx) == 0:
        ax.text(0.5, 0.5, f'{mname} not found', transform=ax.transAxes,
                ha='center', va='center')
        continue
    expr = adata_img.layers['exprs_arcsinh'][:, midx[0]]

    # Sort so high-expressing cells plotted on top
    sort_idx = np.argsort(expr)
    sc = ax.scatter(
        coords[sort_idx, 0], coords[sort_idx, 1],
        c=expr[sort_idx], cmap='magma',
        s=3, alpha=0.9, rasterized=True,
        vmin=0, vmax=np.percentile(expr, 99)
    )
    plt.colorbar(sc, ax=ax, label='arcsinh expression')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(marker_labels.get(mname, mname), fontsize=11)
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')

plt.suptitle('Protein marker expression in tissue space\n(magma: dark=low, bright=high)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PD1 expression overlaid with cell type context
# PD1+ T cells near tumour = potential immunotherapy targets

pd1_idx = np.where(adata_img.var['marker'] == 'PD1')[0][0]
pd1_expr = adata_img.layers['exprs_arcsinh'][:, pd1_idx]

# PD1+ threshold: top 15% expressing cells
pd1_threshold = np.percentile(pd1_expr, 85)
pd1_pos_mask = pd1_expr >= pd1_threshold
tumor_mask = adata_img.obs['celltypes'] == 'Tumor'

fig, ax = plt.subplots(figsize=(10, 10))

ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.1, color='lightgrey', rasterized=True)
ax.scatter(coords[tumor_mask, 0], coords[tumor_mask, 1],
           s=4, alpha=0.5, color='crimson', label='Tumor', rasterized=True)
ax.scatter(coords[pd1_pos_mask, 0], coords[pd1_pos_mask, 1],
           s=8, alpha=0.9, color='dodgerblue',
           label=f'PD1+ cells (top 15%, n={pd1_pos_mask.sum()})',
           zorder=4, rasterized=True)

ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend(fontsize=10, markerscale=3, loc='lower right')
ax.set_title('Tumour cells (red) vs PD1+ cells (blue)\n'
             'PD1+ T cells near tumour are key immunotherapy targets',
             fontsize=11)
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
plt.tight_layout()
plt.show()

# Which cell types are PD1+?
print("Cell type distribution among PD1+ cells:")
pd1_cts = adata_img.obs.loc[adata_img.obs.index[pd1_pos_mask], 'celltypes'].value_counts()
print(pd1_cts.to_string())

### Exercise 2.2

**Task:** Map **Granzyme B (GrzB)** expression spatially and identify which cell types are GrzB-positive.  
Granzyme B is a cytotoxic molecule released by activated CD8 T cells and NK cells.

**Question:** Are GrzB+ cells located near tumour cells? Does co-localisation with PD1 suggest exhausted or active cytotoxic cells?

In [ ]:
# Your code here
grzb_idx = np.where(adata_img.var['marker'] == 'GrzB')[0][0]
grzb_expr = adata_img.layers['exprs_arcsinh'][:, grzb_idx]
grzb_pos = grzb_expr >= np.percentile(grzb_expr, 85)

print("Cell type distribution among GrzB+ cells:")
print(adata_img.obs.loc[adata_img.obs.index[grzb_pos], 'celltypes'].value_counts())

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.1, color='lightgrey', rasterized=True)
ax.scatter(coords[tumor_mask, 0], coords[tumor_mask, 1],
           s=4, alpha=0.5, color='crimson', label='Tumor')
ax.scatter(coords[grzb_pos, 0], coords[grzb_pos, 1],
           s=10, alpha=0.9, color='darkorange', label='GrzB+ (top 15%)', zorder=4)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend(markerscale=3, fontsize=10)
ax.set_title('GrzB+ cells (cytotoxic activity) near tumour')
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
plt.tight_layout()
plt.show()

---
## Part 5: Quantifying Spatial Infiltration

Visualisation is qualitative. We need numbers to compare images systematically.

**Nearest-neighbour distance**: For each immune cell, compute distance to nearest tumour cell.  
- Short distances = immune cells infiltrating the tumour  
- Large distances = immune cells excluded from the tumour  

We use `scipy.spatial.KDTree` for efficient O(n log n) nearest-neighbour search.

In [ ]:
def compute_min_distances(source_coords, target_coords):
    """For each source cell, find distance to nearest target cell."""
    if len(target_coords) == 0 or len(source_coords) == 0:
        return np.array([])
    tree = KDTree(target_coords)
    distances, _ = tree.query(source_coords, k=1)
    return distances


# Compute: for each immune cell type, distance to nearest tumour cell
tumor_coords = coords[adata_img.obs['celltypes'].values == 'Tumor']
cd8_coords   = coords[adata_img.obs['celltypes'].values == 'CD8']
cd4_coords   = coords[adata_img.obs['celltypes'].values == 'CD4']
mac_coords   = coords[adata_img.obs['celltypes'].values == 'MacCD163']

dist_cd8 = compute_min_distances(cd8_coords, tumor_coords)
dist_cd4 = compute_min_distances(cd4_coords, tumor_coords)
dist_mac = compute_min_distances(mac_coords, tumor_coords)

for name, dist in [('CD8', dist_cd8), ('CD4', dist_cd4), ('MacCD163', dist_mac)]:
    if len(dist) > 0:
        print(f"{name} ({len(dist)} cells):")
        print(f"  Median dist to tumour : {np.median(dist):.1f} um")
        print(f"  Fraction within 20 um : {(dist < 20).mean():.1%}")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

bins = np.linspace(0, 200, 60)
dist_data = [
    (dist_cd8, 'CD8+ T cell', 'royalblue'),
    (dist_cd4, 'CD4+ T cell', 'mediumorchid'),
    (dist_mac, 'MacCD163',    'forestgreen'),
]

# Histogram
for dist, label, color in dist_data:
    if len(dist) > 0:
        axes[0].hist(dist, bins=bins, alpha=0.6, label=label, color=color, density=True)
        axes[0].axvline(np.median(dist), color=color, linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Distance to nearest tumour cell (um)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distance of immune cells to tumour\n(dashed = median)')
axes[0].legend()
axes[0].axvline(x=20, color='red', linestyle=':', alpha=0.5)
axes[0].text(22, axes[0].get_ylim()[1]*0.9, '20 um', color='red', fontsize=9)

# ECDF (cumulative distribution)
for dist, label, color in dist_data:
    if len(dist) > 0:
        sorted_dist = np.sort(dist)
        ecdf = np.arange(1, len(sorted_dist)+1) / len(sorted_dist)
        axes[1].plot(sorted_dist, ecdf, label=label, color=color, linewidth=2)
axes[1].axvline(x=20, color='red', linestyle=':', alpha=0.7, label='20 um threshold')
axes[1].axhline(y=0.5, color='grey', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Distance to nearest tumour cell (um)')
axes[1].set_ylabel('Cumulative fraction of cells')
axes[1].set_title('ECDF of immune-to-tumour distances')
axes[1].legend()
axes[1].set_xlim(0, 200)

plt.suptitle('Immune cell infiltration depth', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 6: Comparing Immune Infiltration Across Images

The median distance from CD8 T cells to tumour is a single-number **infiltration score** per image.  
Low score = immune-infiltrated (hot tumour).  
High score = immune-excluded (cold tumour).

We compute this for all 132 images and compare across cancer types.

In [ ]:
# Infiltration score for all images (~30 seconds)
results = []
all_images = adata.obs['image'].unique()

for img_id in all_images:
    sub = adata[adata.obs['image'] == img_id]
    c = sub.obsm['spatial']
    celltypes = sub.obs['celltypes'].values
    t_coords = c[celltypes == 'Tumor']
    cd8_c    = c[celltypes == 'CD8']

    if len(t_coords) < 5 or len(cd8_c) < 3:
        continue

    dists = compute_min_distances(cd8_c, t_coords)
    results.append({
        'image':      img_id,
        'indication': sub.obs['Indication'].iloc[0],
        'n_cells':    len(sub),
        'n_tumor':    len(t_coords),
        'n_cd8':      len(cd8_c),
        'median_dist': np.median(dists),
        'frac_within20': (dists < 20).mean(),
        'tumor_prop':  (celltypes == 'Tumor').mean(),
        'cd8_prop':    (celltypes == 'CD8').mean(),
    })

infil_df = pd.DataFrame(results)
print(f"Processed {len(infil_df)} images")
print()
print("Mean infiltration by cancer type:")
print(infil_df.groupby('indication')[['median_dist', 'frac_within20']].mean().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ind_order = infil_df.groupby('indication')['median_dist'].median().sort_values().index
palette = sns.color_palette('Set2', len(ind_order))

# Boxplot: median distance by cancer type
sns.boxplot(data=infil_df, x='indication', y='median_dist',
            order=ind_order, palette=palette, ax=axes[0])
sns.swarmplot(data=infil_df, x='indication', y='median_dist',
              order=ind_order, color='black', size=4, alpha=0.6, ax=axes[0])
axes[0].set_xlabel('Cancer type')
axes[0].set_ylabel('Median CD8 to tumour distance (um)')
axes[0].set_title('Immune infiltration by cancer type\n(lower = more infiltrated)')

# Scatter: CD8 abundance vs infiltration depth
for ind, grp in infil_df.groupby('indication'):
    axes[1].scatter(grp['cd8_prop'], grp['median_dist'],
                    label=ind, alpha=0.7, s=50)
axes[1].set_xlabel('CD8+ T cell proportion')
axes[1].set_ylabel('Median CD8 to tumour distance (um)')
axes[1].set_title('CD8 abundance vs infiltration depth')
axes[1].legend(title='Indication', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Visualise the most and least infiltrated images
most_inf = infil_df.loc[infil_df['frac_within20'].idxmax(), 'image']
least_inf = infil_df.loc[infil_df['frac_within20'].idxmin(), 'image']

print(f"Most infiltrated : {most_inf[-55:]}")
v1 = infil_df.loc[infil_df['image']==most_inf, 'frac_within20'].values[0]
print(f"  {v1:.1%} of CD8 cells within 20um of tumour")
print(f"Least infiltrated: {least_inf[-55:]}")
v2 = infil_df.loc[infil_df['image']==least_inf, 'frac_within20'].values[0]
print(f"  {v2:.1%} of CD8 cells within 20um of tumour")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, img_id, title in zip(
    axes,
    [most_inf, least_inf],
    ['Most infiltrated', 'Least infiltrated (immune-excluded)']
):
    sub = adata[adata.obs['image'] == img_id]
    c_sub = sub.obsm['spatial']
    ind = sub.obs['Indication'].iloc[0]
    celltypes_sub = sub.obs['celltypes'].values

    bg = ~np.isin(celltypes_sub, ['Tumor', 'CD8'])
    ax.scatter(c_sub[bg, 0], c_sub[bg, 1], s=2, alpha=0.1, color='lightgrey', rasterized=True)
    for ct, col in [('Tumor', 'crimson'), ('CD8', 'royalblue')]:
        m = celltypes_sub == ct
        ax.scatter(c_sub[m, 0], c_sub[m, 1], s=5, alpha=0.8, color=col,
                   label=ct, rasterized=True)

    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(f'{title}\n[{ind}]  {img_id[-40:]}', fontsize=10)
    ax.legend(markerscale=3, fontsize=10, loc='lower right')
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')

plt.suptitle('Visual comparison: hot vs cold tumour phenotype', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 2.3: Reusable Spatial Visualisation Function

**Task:** Write a function `plot_tissue(adata, image_id, marker=None, highlight_types=None)` that:
1. Plots all cells coloured by cell type
2. Optionally overlays expression of a specific marker as colour
3. Optionally highlights only specific cell types (rest grey)

Use it to compare two images from different cancer indications.

In [ ]:
def plot_tissue(adata, image_id, marker=None, highlight_types=None, figsize=(10, 10)):
    """
    Spatial plot of a single IMC image.

    Parameters
    ----------
    adata          : full AnnData with obsm['spatial'] and layers['exprs_arcsinh']
    image_id       : string image ID
    marker         : optional marker name to colour cells by expression
    highlight_types: list of cell types to show in colour (rest = grey)
    """
    sub = adata[adata.obs['image'] == image_id]
    c   = sub.obsm['spatial']
    ind = sub.obs['Indication'].iloc[0] if 'Indication' in sub.obs.columns else ''

    fig, ax = plt.subplots(figsize=figsize)

    if marker is not None:
        midx = np.where(sub.var['marker'] == marker)[0]
        if len(midx) == 0:
            raise ValueError(f"Marker '{marker}' not found")
        expr = sub.layers['exprs_arcsinh'][:, midx[0]]
        sort_idx = np.argsort(expr)
        sc = ax.scatter(
            c[sort_idx, 0], c[sort_idx, 1],
            c=expr[sort_idx], cmap='viridis', s=4, alpha=0.9,
            rasterized=True, vmin=0, vmax=np.percentile(expr, 99)
        )
        plt.colorbar(sc, ax=ax, label=f'{marker} (arcsinh)')
        ax.set_title(f'{marker} expression | {image_id[-40:]} [{ind}]', fontsize=10)
    else:
        cts = sorted(sub.obs['celltypes'].unique())
        pal = dict(zip(cts, sns.color_palette('tab20', len(cts))))
        celltypes_sub = sub.obs['celltypes'].values

        if highlight_types:
            bg = ~np.isin(celltypes_sub, highlight_types)
            ax.scatter(c[bg, 0], c[bg, 1], s=2, alpha=0.15, color='lightgrey', rasterized=True)
            for ct in highlight_types:
                if ct in pal:
                    m = celltypes_sub == ct
                    ax.scatter(c[m, 0], c[m, 1], s=5, alpha=0.8,
                               color=pal[ct], label=ct, rasterized=True)
        else:
            for ct in cts:
                m = celltypes_sub == ct
                ax.scatter(c[m, 0], c[m, 1], s=3, alpha=0.7,
                           color=pal[ct], label=ct, rasterized=True)

        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8,
                  markerscale=3, title='Cell type')
        ax.set_title(f'Cell types | {image_id[-40:]} [{ind}]', fontsize=10)

    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')
    plt.tight_layout()
    return fig


# Compare BREAS vs HN image
breas_imgs = adata.obs.query('Indication == "BREAS"')['image'].unique()
hn_imgs    = adata.obs.query('Indication == "HN"')['image'].unique()

_ = plot_tissue(adata, breas_imgs[0], highlight_types=['Tumor', 'CD8'])
plt.suptitle('Breast cancer image', y=1.01)
plt.show()

_ = plot_tissue(adata, hn_imgs[0], highlight_types=['Tumor', 'CD8'])
plt.suptitle('Head & Neck cancer image', y=1.01)
plt.show()

---
## Summary

| Concept | Key takeaway |
|---|---|
| `obsm['spatial']` | Required for spatial tools; created from `obs['Pos_X/Y']` |
| Cell type overlay | Reveals tissue architecture - tumour islands, immune stroma |
| Marker expression mapping | Spatial patterns of PD1, GrzB, Ecad reveal functional states |
| KD-tree distances | Efficient O(n log n) nearest-neighbour search for infiltration |
| Infiltration score | Median CD8 to tumour distance - single number per image |

**Biological insight:** Immune infiltration varies dramatically across cancer types and patients.  
This spatial heterogeneity is a key predictor of immunotherapy response.

**Next:** Tutorial 3 uses Squidpy to statistically test which cell types are enriched as spatial neighbours - moving from visualisation to rigorous hypothesis testing.

---